In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("ycsb_results.csv").drop(columns=["filename"])
df

In [ ]:
groups_raw = df.drop(columns=["threads", "workload"]).groupby(["replication_factor","consistency"])
groups = groups_raw.mean()
groups

In [ ]:
groups = groups.reset_index()

In [ ]:
#Not every workload has the same columns. This is to show which ones do.
#It's doubtfull we'll use most of these, but it's still good to know
workloads = {w: df[df["workload"] == w].dropna(axis=1).columns for w in "abcdef"}
workloads

In [ ]:
figure_id=0
def _next_figure_name():
    global figure_id
    figure_id += 1
    return f"Figure {figure_id:02d}.png" 


def simple_graph(filter_tuples, grouper, title, values, legend, y_label=None):
    data = df
    for key, value in filter_tuples:
        data = data[data[key] == value]

    data = data.drop(columns=[key for key,_ in filter_tuples])
    data = grouper(data.dropna(axis=1))
    data = data[values]
    ax = data.plot.bar(rot=0, grid=True, ylabel=y_label)
    ax.legend(legend, bbox_to_anchor=(1.05,1), loc="best")
    plt.title(title)
    plt.savefig(_next_figure_name(), bbox_inches="tight")
    plt.show()

def complex_graph(filter_tuples, title, main_index, sub_index, values, legend, y_label=None, grouper=None):
    data = df
    for key, value in filter_tuples:
        data = data[data[key] == value]

    data = data.drop(columns=[key for key,_ in filter_tuples])
    if grouper is not None: data = grouper(data)
    
    ax = data.pivot(index=main_index, columns=sub_index, values=values).plot.bar(rot=0, grid=True, ylabel=y_label)
    ax.legend(legend, bbox_to_anchor=(1.05,1), loc="best")
    plt.title(title)
    plt.savefig(_next_figure_name(), bbox_inches="tight")
    plt.show()
    

In [ ]:
filters = (("replication_factor",5), ("workload","a"))
y = ["update_maxlatency_us","update_95thpercentilelatency_us","update_averagelatency_us"]
y_legend = ["Max", "95th %", "Avg"]

grouper = lambda table: table.groupby("consistency").mean()

simple_graph(filters, grouper, "Update Latencies for Replication Factor 5 (Workload A)", y, y_legend, "Microseconds")
plt.show()

In [ ]:
filters = (("replication_factor",5),)
y=["overall_throughput_ops_sec"]
y_legend=["Workload A", "Workload B", "Workload C", "Workload D","Workload E","Workload F"]

grouper = lambda table: table.groupby(["consistency","workload"]).mean().reset_index()

complex_graph(filters, "Overall Throughputs (Replication 5)", "consistency", "workload", y, y_legend, "Operations per Second", grouper)